[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P19/P19-2_Etude_Projet_CORRIGE.ipynb)

# Étude V : suivi de production

Étude de synthèse de la partie V du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

Les chapitres 17 à 19 forment une chaîne : explorer, préparer, ajuster, juger, mettre en forme pour un réseau. L'étude la rejoue d'un bout à l'autre sur le suivi de production de vingt mille machines, avec un projet de régression : prédire l'indice de santé, de $0$ à $100$, à partir des mesures. Le notebook `P19-2` du dépôt reprend chaque étape. La figure du chapitre déroule la chaîne.

Jeu `Suivi_Production_Machines` : les quatorze colonnes, dont `indice_sante_machine_0_100`, la cible.

Quatre phrases cadrent le projet. L'objectif terminal : prévoir l'indice de santé d'une machine. L'objectif instrumental : une régression linéaire sur les mesures. Le critère de résultat : le $R^2$ sur la base de test. La vérification : le test est mis de côté avant tout croisement avec la cible, et ouvert une seule fois, à la fin. L'état de santé, en quatre classes, est calculé à partir de l'indice : le prédire serait recopier la cible. De même la classe de productivité, qui découle de la productivité. Les deux colonnes de texte restantes, presque constantes, ne disent rien.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-Suivi_Production_Machines.xlsx` | 20000 | 14 | ancienneté en années, complexité et efficacité sans dimension, nombre de capteurs entier, utilisation en proportion, température en °C, vibrations en mm/s, consommation en kWh/h, productivité en pièces/h, indice de santé de 0 à 100 |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [1]:
from pathlib import Path
import urllib.request

FICHIER = "MecaIA-Suivi_Production_Machines.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-Suivi_Production_Machines.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeu lu dans le dossier des données :", FICHIER)

jeu lu dans le dossier des données : MecaIA-Suivi_Production_Machines.xlsx


### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [2]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("pandas", version("pandas"))
print("scikit-learn", version("scikit-learn"))

Python 3.12.0
numpy 2.4.6
pandas 3.0.5
scikit-learn 1.9.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [3]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [4]:
def taux_manquants(df):
    """Taux de valeurs manquantes par colonne, en pourcentage, trié par ordre décroissant.

    >>> d = pd.DataFrame({"a": [1.0, None, None, 4.0], "b": [1.0, 2.0, 3.0, None], "c": [1, 2, 3, 4]})
    >>> taux_manquants(d).round(1).to_dict()
    {'a': 50.0, 'b': 25.0, 'c': 0.0}
    """
    return (df.isna().mean() * 100).sort_values(ascending=False)

### Préparation

In [5]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
pd.set_option("display.width", 100)

## Étape 1 : cahier des charges

Cadrer. Objectif terminal : prédire l'indice de santé d'une machine ; critère de résultat : R² sur le test.

Les contrôles de la cellule lisent les variables `df`, `entrees`, `maxi_precedent`, `mini_suivant` : gardez ces noms.

In [6]:
# Solution :
data = pd.read_excel(DONNEES / "MecaIA-Suivi_Production_Machines.xlsx")
df = data.copy()
# Fin de la solution
assert df.shape == (20000, 14), "la fiche du jeu dit la même taille"
# Solution :
cible = "indice_sante_machine_0_100"
# L'état de santé est calculé à partir de l'indice (variable déterminée a posteriori), la classe de productivité à
# partir de la productivité : on les écarte. Les deux colonnes de texte restantes sont presque constantes.
ecartees = ["etat_sante_machine", "classe_niveau_productivite", "type_de_machine", "mode_de_fonctionnement"]
entrees = [c for c in df.columns if c not in ecartees + [cible]]
print(f"cible : {cible} ; {len(entrees)} entrées numériques ; {len(ecartees)} colonnes écartées")
# Fin de la solution
assert len(entrees) == 9 and df["type_de_machine"].value_counts(normalize=True).iloc[0] > 0.99
# Solution :
bornes = df.groupby("etat_sante_machine")[cible].agg(["min", "max"])
print(bornes.round(1).sort_values("max").to_string())
ordre_etats = bornes.sort_values("max")
mini_suivant = ordre_etats["min"].iloc[1:].to_numpy()
maxi_precedent = ordre_etats["max"].iloc[:-1].to_numpy()
# Fin de la solution
assert (
    (mini_suivant >= maxi_precedent - 1e-9).all()
), "les intervalles d'indice des états ne se chevauchent pas"
print("Étape 1 : contrôles passés")

cible : indice_sante_machine_0_100 ; 9 entrées numériques ; 4 colonnes écartées
                     min    max
etat_sante_machine             
Panne_imminente      0.0   32.0
Degradation_severe  32.1   42.0
Degradation_legere  42.1   64.0
RAS                 64.1  100.0
Étape 1 : contrôles passés


## Étape 2 : découpage et exploration

Découpage et exploration. Le test est mis de côté avant tout calcul qui croise la cible.

Les contrôles de la cellule lisent les variables `apprentissage`, `correlations`, `taux`, `test` : gardez ces noms.

In [7]:
# Solution :
apprentissage, test = train_test_split(df, test_size=0.2, random_state=0)
X_train, y_train = apprentissage[entrees], apprentissage[cible]
X_test, y_test = test[entrees], test[cible]
print(f"apprentissage {X_train.shape}, test {X_test.shape}")
# Fin de la solution
assert (
    len(apprentissage) == 16000 and len(test) == 4000 and len(set(apprentissage.index) & set(test.index)) == 0
)
# Solution :
# Suite de l'étape 2 : explorer sur l'apprentissage seul.
print("explorer l'apprentissage")
taux = taux_manquants(X_train)
correlations = X_train.corrwith(y_train).sort_values()
print(taux.head(4).round(2).to_string())
print(correlations.round(2).head(3).to_string())
# Fin de la solution
assert (
    (taux > 0).sum() == 3 and taux.iloc[0] < 10.5
), "trois colonnes de capteurs ont près de 10 % de vides"
assert correlations.index[0] == "niveau_vibrations_en_mm_par_s" and correlations.iloc[0] < -0.75
assert (
    abs(correlations.get("taux_utilisation_machine")) < 0.1
), "l'utilisation ne dit rien de la santé"
print("Étape 2 : contrôles passés")

apprentissage (16000, 9), test (4000, 9)
explorer l'apprentissage
niveau_vibrations_en_mm_par_s               10.17
consommation_electrique_en_kWh_par_heure    10.14
temperature_fonctionnement_en_degC           9.67
anciennete_machine_en_annees                 0.00
niveau_vibrations_en_mm_par_s        -0.91
temperature_fonctionnement_en_degC   -0.87
anciennete_machine_en_annees         -0.61
Étape 2 : contrôles passés


## Étape 3 : préparation et ajustement

Préparer sans fuite. L'imputation et la mise à l'échelle apprennent sur l'apprentissage seul.

Les contrôles de la cellule lisent les variables `apprises`, `ecart_fuite`, `medianes_apprentissage` : gardez ces noms.

In [8]:
# Solution :
chaine = Pipeline([("imputation", SimpleImputer(strategy="median")),
                   ("echelle", StandardScaler()),
                   ("modele", LinearRegression())])
chaine.fit(X_train, y_train)
medianes_apprentissage = X_train.median()
medianes_tout = df[entrees].median()
apprises = chaine.named_steps["imputation"].statistics_
# Fin de la solution
assert (
    np.allclose(apprises, medianes_apprentissage.to_numpy())
), "la médiane apprise est celle de l'apprentissage"
# Solution :
ecart_fuite = float(np.abs(medianes_apprentissage.to_numpy() - medianes_tout.to_numpy()).max())
print(f"écart maximal entre les médianes de l'apprentissage et celles du jeu entier : {ecart_fuite:.3f}")
# Fin de la solution
assert ecart_fuite > 0.0, "préparer sur le jeu entier changerait les valeurs : c'est la fuite"
# Solution :
# la préparation fautive : médiane et échelle apprises sur tout le jeu, avant le découpage
imputeur_fuite = SimpleImputer(strategy="median").fit(df[entrees])
echelle_fuite = StandardScaler().fit(imputeur_fuite.transform(df[entrees]))
modele_fuite = LinearRegression().fit(echelle_fuite.transform(imputeur_fuite.transform(X_train)), y_train)
r2_fuite = modele_fuite.score(echelle_fuite.transform(imputeur_fuite.transform(X_test)), y_test)
# Fin de la solution
print("Étape 3 : contrôles passés")

écart maximal entre les médianes de l'apprentissage et celles du jeu entier : 0.100
Étape 3 : contrôles passés


## Étape 4 : évaluation du modèle

Un premier modèle, jugé une seule fois sur le test.

Les contrôles de la cellule lisent les variables `coefficients`, `r2_test`, `r2_train` : gardez ces noms.

In [9]:
# Solution :
r2_train = chaine.score(X_train, y_train)
r2_test = chaine.score(X_test, y_test)
rmse_test = root_mean_squared_error(y_test, chaine.predict(X_test))
coefficients = pd.Series(chaine.named_steps["modele"].coef_, index=entrees).sort_values()
print(f"R2 apprentissage {r2_train:.3f}, R2 test {r2_test:.3f}, RMSE test {rmse_test:.2f} points d'indice")
print(f"R2 test avec la préparation fautive : {r2_fuite:.3f} (écart {abs(r2_fuite - r2_test):.4f})")
print("coefficients les plus négatifs :", coefficients.round(2).head(3).to_dict())
# Fin de la solution
assert abs(r2_score(y_test, chaine.predict(X_test)) - r2_test) < 1e-12
assert (
    r2_test > 0.6 and abs(r2_train - r2_test) < 0.03
), "le modèle généralise : peu d'écart entre apprentissage et test"
assert (
    coefficients.iloc[0] < -5 and "niveau_vibrations_en_mm_par_s" in coefficients.index[:2]
), "les vibrations pèsent parmi les premières entrées"
assert (
    abs(r2_fuite - r2_test) < 0.01
), "ici la fuite est discrète, mais le principe reste de ne pas l'employer"
print("Étape 4 : contrôles passés")

R2 apprentissage 0.862, R2 test 0.859, RMSE test 8.33 points d'indice
R2 test avec la préparation fautive : 0.859 (écart 0.0000)
coefficients les plus négatifs : {'niveau_vibrations_en_mm_par_s': -9.18, 'consommation_electrique_en_kWh_par_heure': -8.01, 'temperature_fonctionnement_en_degC': -6.95}
Étape 4 : contrôles passés


## Étape 5 : entrées d'un réseau

Des données prêtes pour un réseau de neurones, mises à l'échelle avec les statistiques de l'apprentissage.

Les contrôles de la cellule lisent les variables `X_test_n`, `X_train_n` : gardez ces noms.

In [10]:
# Solution :
imputeur = chaine.named_steps["imputation"]
X_train_n = chaine.named_steps["echelle"].transform(imputeur.transform(X_train)).astype("float32")
X_test_n = chaine.named_steps["echelle"].transform(imputeur.transform(X_test)).astype("float32")
print(f"tableaux pour le réseau : {X_train_n.shape} et {X_test_n.shape}, type {X_train_n.dtype}")
# Fin de la solution
assert (
    not np.isnan(X_train_n).any() and not np.isnan(X_test_n).any()
), "plus aucune valeur manquante"
assert np.abs(X_train_n.mean(axis=0)).max() < 1e-4, "l'apprentissage est centré"
assert (
    np.abs(X_test_n.mean(axis=0)).max() > 1e-4
), "le test ne l'est pas : il n'a pas servi aux statistiques"
# Solution :
print("étude V vérifiée")
# Fin de la solution
print("Étape 5 : contrôles passés")

tableaux pour le réseau : (16000, 9) et (4000, 9), type float32
étude V vérifiée
Étape 5 : contrôles passés
